## Note

There are several libraries available to calculate **metrics such as Precision, Recall, and F1 Score**,  
but in this notebook, we are using **`sklearn.metrics`**.  
A working example is provided below.



In [2]:
# Step 1: Import required libraries pandas, re, sklearn.metrics
import pandas as pd
import re
from sklearn.metrics import precision_score, recall_score, f1_score

In [3]:
# Step 2: Load the CSV dataset
questions_df = pd.read_csv("Questions.csv")
questions_df.head()

,Question,Expected Answer,Actual Answer
0,What is the capital of France?,The capital of France is Paris,The capital of France is Paris
1,Which elements were discovered by Marie Curie?,Marie Curie discovered two elements: 1. Poloni...,"Marie Curie, along with her husband Pierre Cur..."
2,By whom Pride and Prejudice was written?,Pride and Prejudice was written by Jane Austen,Pride and Prejudice was written by Jane Austen
3,What is the square root of 81?,The square root of 81 is 9,The square root of 81 is 9
4,Who won the World Cup in 1998?,Can you please tell me what\ntype of competiti...,"France won the FIFA World Cup in 1998, held in..."


### How to Calculate Tokens

There are two main ways to calculate tokens:

1. **Manual (via OpenAI Tokenizer Tool or other calculator)**  
   - You can use the [OpenAI Tokenizer](https://platform.openai.com/tokenizer) to paste text and see exactly how it will be split into tokens for different models.  
   - This method is quick and requires no coding, but it is manual.

2. **Programmatic (via `tiktoken` (OpenAI) library or simmilar ones)**  
   - OpenAI provides the [`tiktoken`](https://github.com/openai/tiktoken) library, which allows you to calculate tokens directly in code.  
   - Example in Python:

     ```python
     import tiktoken

     # Choose encoding for your model
     encoding = tiktoken.encoding_for_model("gpt-4")

     text = "Hello world!"
     tokens = encoding.encode(text)

     print("Number of tokens:", len(tokens))
     ```

---
- You can **compare results** from the [Tokenizer UI](https://platform.openai.com/tokenizer) and the `tiktoken` library.  
- Both should give the same token count for the same model and text.  
- This is useful if you want to double-check correctness or validate your implementation.

---

### Your Task

- Calculate tokens for **expected** and **actual answers**.  
- Add two new attributes to `questions_df`:  
  - `Expected_answers_tokens`  
  - `Actual_answers_tokens`  
- We suggest using **`tiktoken`** to measure token usage and ensure consistency between the model’s input and outputs.



In [4]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4")

questions_df["Expected_answers_tokens"] = questions_df['Expected Answer'].apply(lambda x: len(encoding.encode(x)))
questions_df["Actual_answers_tokens"] = questions_df['Actual Answer'].apply(lambda x: len(encoding.encode(x)))

In [56]:
questions_df

,Question,Expected Answer,Actual Answer,Expected_answers_tokens,Actual_answers_tokens
0,What is the capital of France?,The capital of France is Paris,The capital of France is Paris,6,6
1,Which elements were discovered by Marie Curie?,Marie Curie discovered two elements: 1. Poloni...,"Marie Curie, along with her husband Pierre Cur...",23,136
2,By whom Pride and Prejudice was written?,Pride and Prejudice was written by Jane Austen,Pride and Prejudice was written by Jane Austen,12,12
3,What is the square root of 81?,The square root of 81 is 9,The square root of 81 is 9,9,9
4,Who won the World Cup in 1998?,Can you please tell me what\ntype of competiti...,"France won the FIFA World Cup in 1998, held in...",24,14
5,Lobamba is capital of?,"Lobamba is historical capital of Eswatini, but...",I couldn't find any information on a well-know...,23,53
6,Which planet is the largest in the System?,"If you meant our solar system, the largest pla...",The largest planet in our solar system is Jupiter,12,9


## Task: Evaluate Responses with Precision, Recall and F1 Score

Now that we can see where tokens of expected and actual answers differ,  but it just a pointing - it does not mean that outputs are incorrect.
It is useful to have detailed measurements of **Precision, Recall, and F1 Score** to evaluate responses.

We will use **`sklearn.metrics`** for this purpose.  
Here is an example showing how to calculate **Precision** for a single question:

## Example: Calculating Precision for a Single Question

We want to evaluate how well the model's answer matches the expected answer.  
Here, we use **Precision** as a metric, which measures the proportion of relevant tokens in the predicted answer.

**Question:** Who invented the telephone?  
- **Expected Answer:** Alexander Graham Bell  
- **Actual Answer:** Bell  

**Steps in the calculation:**
1. Tokenize both expected and actual answers.  
2. Create a combined list of all unique tokens.  
3. Convert each answer into a binary vector, indicating the presence (1) or absence (0) of each token.  
4. Use `sklearn.metrics.precision_score` to calculate precision.
[Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html#sklearn.metrics.precision_score) 
```python
from sklearn.metrics import precision_score 

# Step 1: Tokenization
expected_tokens = "Alexander Graham Bell".lower().split()
['alexander', 'graham', 'bell']

actual_tokens = "Bell".lower().split()
['bell']

# Step 2: Create combined list of unique tokens
all_tokens = list(set(expected_tokens).union(set(actual_tokens)))
['alexander', 'graham', 'bell']

# Step 3: Convert to binary representation for each elemt in the combined list of unique tokens and check their presence in expected_tokens and actual_tokens
# 1 if the token is present in the answer, 0 otherwise
y_true = [1 if token in expected_tokens else 0 for token in all_tokens]
[1, 1, 1] # means that all tokens from all_tokens present in expected_tokens
y_pred = [1 if token in actual_tokens else 0 for token in all_tokens]
[0, 0, 1] # means that only 1 last token from all_tokens present in actual_tokens

# Step 4: Calculate Precision
precision = precision_score(y_true, y_pred, zero_division=0)
print("Precision:", precision)
Precision: 1.0






### Your task 
is to calculate **Precision, Recall, and F1 Score** for all questions and add corresponding attributes to `questions_df`:
- `Precision`  
- `Recall`  
- `F1_Score`

After calculating tokens and evaluation metrics, the `questions_df` should have the following columns:

| Column                     | Description |
|-----------------------------|-------------|
| `Question`                  | The question text |
| `Expected Answer`           | The expected answer text |
| `Actual Answer`             | The model’s actual answer text |
| `Expected_answers_tokens`   | Number of tokens in the expected answer |
| `Actual_answers_tokens`     | Number of tokens in the actual answer |
| `Precision`                 | Precision score of the actual answer vs. expected answer |
| `Recall`                    | Recall score of the actual answer vs. expected answer |
| `F1_Score`                  | F1 Score of the actual answer vs. expected answer | 

In [ ]:
# OPTION 1 with adding and dropping intermediate columns

def get_labels(row):
    expected = set(row["expected_tokens"])
    actual = set(row["actual_tokens"])
    all_tokens = row["all_tokens"]
    y_true = [1 if token in expected else 0 for token in all_tokens]
    y_pred = [1 if token in actual else 0 for token in all_tokens]
    return y_true, y_pred

# Tokenize answers
questions_df["expected_tokens"] = questions_df["Expected Answer"].str.lower().str.split()
questions_df["actual_tokens"] = questions_df["Actual Answer"].str.lower().str.split()

# Get all unique tokens
questions_df["all_tokens"] = questions_df.apply(lambda row: list(set(row["expected_tokens"]).union(row["actual_tokens"])), axis=1)
questions_df[["y_true", "y_pred"]] = questions_df.apply(lambda row: pd.Series(get_labels(row)), axis=1)

# Calculate metrics
questions_df["Precision"] = questions_df.apply(lambda row: round(precision_score(row["y_true"], row["y_pred"], zero_division=0), 2), axis=1)
questions_df["Recall"] = questions_df.apply(lambda row: round(recall_score(row["y_true"], row["y_pred"], zero_division=0), 2), axis=1)
questions_df["F1_Score"] = questions_df.apply(lambda row: round(f1_score(row["y_true"], row["y_pred"], zero_division=0), 2), axis=1)
questions_df.drop(columns=["expected_tokens", "actual_tokens", "all_tokens", "y_true", "y_pred"], inplace=True)


In [ ]:
questions_df

In [ ]:
# OPTION 2 without adding and dropping intermediate columns

def compute_metrics(row):
    expected_tokens = set(row["Expected Answer"].lower().split())
    actual_tokens = set(row["Actual Answer"].lower().split())
    all_tokens = list(expected_tokens.union(actual_tokens))
    y_true = [1 if token in expected_tokens else 0 for token in all_tokens]
    y_pred = [1 if token in actual_tokens else 0 for token in all_tokens]
    precision = round(precision_score(y_true, y_pred, zero_division=0), 2)
    recall = round(recall_score(y_true, y_pred, zero_division=0), 2)
    f1 = round(f1_score(y_true, y_pred, zero_division=0), 2)
    return pd.Series({"Precision": precision, "Recall": recall, "F1_Score": f1})

questions_df[["Precision", "Recall", "F1_Score"]] = questions_df.apply(compute_metrics, axis=1)

In [6]:
questions_df

,Question,Expected Answer,Actual Answer,Expected_answers_tokens,Actual_answers_tokens,Precision,Recall,F1_Score
0,What is the capital of France?,The capital of France is Paris,The capital of France is Paris,6,6,1.00,1.00,1.00
1,Which elements were discovered by Marie Curie?,Marie Curie discovered two elements: 1. Poloni...,"Marie Curie, along with her husband Pierre Cur...",23,136,0.10,0.55,0.17
2,By whom Pride and Prejudice was written?,Pride and Prejudice was written by Jane Austen,Pride and Prejudice was written by Jane Austen,12,12,1.00,1.00,1.00
3,What is the square root of 81?,The square root of 81 is 9,The square root of 81 is 9,9,9,1.00,1.00,1.00
4,Who won the World Cup in 1998?,Can you please tell me what\ntype of competiti...,"France won the FIFA World Cup in 1998, held in...",24,14,0.11,0.06,0.08
5,Lobamba is capital of?,"Lobamba is historical capital of Eswatini, but...",I couldn't find any information on a well-know...,23,53,0.12,0.36,0.19
6,Which planet is the largest in the System?,"If you meant our solar system, the largest pla...",The largest planet in our solar system is Jupiter,12,9,0.78,0.64,0.70
